In [1]:
from pathlib import Path
import os
import time
import random
import torch

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_MLP import SingleOutMLPNet
from metrics import eval_single_metrics_test

import pickle    

In [2]:
OUTDIR = Path("./Results_MLP")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path("./trained_models_MLP")
MODELSDIR.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

#### Slice data

In [7]:
scores_df = pd.read_csv("Results_MLP/shap_values/tables/shap_scores.csv")

top_k = 25
selector = scores_df["original_position"][:top_k].to_numpy()
sorted_selector = np.sort(selector)

kept_features = scores_df["feature"][selector].values.tolist()
with open(f"data_processed/dataframes/MLP/input_cols_pruned_shap{top_k}.pkl", "wb") as f:
    pickle.dump(kept_features, f)

X_train_sliced = X_train[:, sorted_selector]
X_val_sliced = X_val[:, sorted_selector]
X_test_sliced = X_test[:, sorted_selector]

X_train_s_sliced = X_train_s[:, sorted_selector]
X_val_s_sliced = X_val_s[:, sorted_selector]
X_test_s_sliced = X_test_s[:, sorted_selector]

#### Dataloaders

In [8]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s_sliced, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s_sliced, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s_sliced, y_test_s, BATCH_SIZE)

### Models and training

In [9]:
EPOCHS = 150
METRICS_EVERY = int(EPOCHS/30)
# METRICS_EVERY = 1
LR = 1e-3
WD = 1e-6

In [10]:
IN_DIM = X_train_s_sliced.shape[1]
HIDDEN = (512, 256, 128)

In [11]:
torch.manual_seed(SEED)    
model_single = SingleOutMLPNet(IN_DIM, hidden=HIDDEN).to(DEVICE)    
loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(OUTDIR / f"train_history/train_history_shap{top_k}.csv", 
               index=False)
torch.save(model_single.state_dict(), MODELSDIR / f"MLP_model_shap{top_k}.pt")

ep=001 tr_loss=0.2168 | va_loss=0.1860 | tr R2=0.615 | va R2=0.593 | tr RMSE=0.620 | va RMSE=0.624 | tr MAE=0.486 | va MAE=0.492 | lr=1.00e-03 | bad_epochs=0
ep=005 tr_loss=0.1638 | va_loss=0.1777 | tr R2=0.683 | va R2=0.610 | tr RMSE=0.563 | va RMSE=0.611 | tr MAE=0.437 | va MAE=0.477 | lr=1.00e-03 | bad_epochs=0
ep=010 tr_loss=0.1394 | va_loss=0.1818 | tr R2=0.728 | va R2=0.599 | tr RMSE=0.521 | va RMSE=0.619 | tr MAE=0.404 | va MAE=0.480 | lr=1.00e-03 | bad_epochs=3
ep=015 tr_loss=0.1147 | va_loss=0.1820 | tr R2=0.790 | va R2=0.599 | tr RMSE=0.458 | va RMSE=0.619 | tr MAE=0.358 | va MAE=0.485 | lr=1.00e-03 | bad_epochs=4
ep=020 tr_loss=0.0845 | va_loss=0.1947 | tr R2=0.848 | va R2=0.567 | tr RMSE=0.390 | va RMSE=0.644 | tr MAE=0.301 | va MAE=0.501 | lr=1.00e-03 | bad_epochs=9
ep=025 tr_loss=0.0472 | va_loss=0.2000 | tr R2=0.910 | va R2=0.553 | tr RMSE=0.301 | va RMSE=0.654 | tr MAE=0.224 | va MAE=0.509 | lr=1.00e-04 | bad_epochs=14
Early stopping at epoch 26. Best epoch was 11 with 

In [12]:
total_time = np.sum(hist_df["time"].to_numpy())
print(f"Total training time: {total_time} sec.")

Total training time: 10.491286277770996 sec.


In [13]:
m_te = eval_single_metrics_test(model_single, X_test_s_sliced, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [14]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)
single_metrics_path = OUTDIR / f"test_metrics/test_metrics_shap{top_k}.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [15]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.641188,8.169467,6.39218
